In [43]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler, PowerTransformer, OrdinalEncoder
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.metrics import r2_score,mean_absolute_error

In [2]:
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna
import dagshub
import mlflow

c:\Users\Jay Kanakia\Desktop\CampusX\Projects\swiggy_delivery_time_prediction\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from sklearn import set_config
set_config(transform_output='pandas')

In [4]:
# Load the data
df = pd.read_csv(r'C:\Users\Jay Kanakia\Desktop\CampusX\Projects\swiggy_delivery_time_prediction\notebooks\data_eda.csv')
df.sample(2)

,rider_id,age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,order_date,weather,traffic,...,city_name,order_day,order_month,order_day_of_week,is_weekend,pickup_time_minutes,order_time_hour,order_time_of_day,distance,distance_type
29964,JAPRES06DEL02,36.0,4.2,26.911927,75.797282,26.961927,75.847282,2022-03-13,fog,jam,...,JAP,13,3,sunday,1,5.0,20.0,evening,7.448364,medium
42622,COIMBRES04DEL01,20.0,4.8,11.024839,77.007003,11.064839,77.047003,2022-03-15,windy,medium,...,COIMB,15,3,tuesday,0,15.0,15.0,afternoon,6.232153,medium


In [5]:
# dropping unnecessary columns

df.drop(columns=['rider_id',
                    'restaurant_latitude',
                    'restaurant_longitude',
                    'delivery_latitude',
                    'delivery_longitude',
                    'order_date',
                    "order_time_hour",
                    "order_day",
                    "city_name",
                    "order_day_of_week",
                    "order_month"],inplace=True)

In [6]:
df.sample(2)

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
19377,38.0,4.1,sandstorms,medium,0,drinks,motorcycle,2.0,no,metropolitian,40,0,10.0,afternoon,10.587733,long
33127,26.0,4.8,sandstorms,low,2,snack,motorcycle,0.0,no,metropolitian,16,0,5.0,night,4.656991,short


In [7]:
df.isnull().sum()

age                    1854
ratings                1908
weather                 525
traffic                 510
vehicle_condition         0
type_of_order             0
type_of_vehicle           0
multiple_deliveries     993
festival                228
city_type              1198
time_taken                0
is_weekend                0
pickup_time_minutes    1640
order_time_of_day      2070
distance               3630
distance_type          3630
dtype: int64

In [8]:
missing_col = df.isnull().any(axis=0).loc[lambda x : x].index
missing_col

Index(['age', 'ratings', 'weather', 'traffic', 'multiple_deliveries',
       'festival', 'city_type', 'pickup_time_minutes', 'order_time_of_day',
       'distance', 'distance_type'],
      dtype='object')

# Dropping missing values

In [9]:
temp_df = df.copy().dropna()
temp_df.sample(2)

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
23024,39.0,4.9,sandstorms,jam,2,buffet,motorcycle,1.0,no,metropolitian,26,0,5.0,night,9.192333,medium
25154,36.0,4.6,cloudy,jam,0,meal,motorcycle,0.0,no,metropolitian,34,0,15.0,evening,7.761839,medium


In [10]:
temp_df.shape

(37695, 16)

In [11]:
temp_df.isnull().sum()

age                    0
ratings                0
weather                0
traffic                0
vehicle_condition      0
type_of_order          0
type_of_vehicle        0
multiple_deliveries    0
festival               0
city_type              0
time_taken             0
is_weekend             0
pickup_time_minutes    0
order_time_of_day      0
distance               0
distance_type          0
dtype: int64

In [12]:
X = temp_df.drop(columns = ['time_taken'])
y = temp_df['time_taken']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

X_train.shape,y_test.shape

((30156, 15), (7539,))

In [13]:
# target column transform

pt = PowerTransformer(method='yeo-johnson')

y_train_pt = pt.fit_transform(y_train.values.reshape(-1,1))
y_test_pt = pt.transform(y_test.values.reshape(-1,1))

y_train_pt

,x0
0,2.028672
1,0.554539
2,-2.024267
3,-0.173699
4,0.554539
...,...
30151,0.457580
30152,-0.173699
30153,-1.350937
30154,0.047111


## Preprocessing

In [14]:
# do basic preprocessing

num_cols = ["age","ratings","pickup_time_minutes","distance"]

nominal_cat_cols = ['weather','type_of_order',
                    'type_of_vehicle',"festival",
                    "city_type",
                    "is_weekend",
                    "order_time_of_day"]

ordinal_cat_cols = ["traffic","distance_type"]

In [15]:
for col in ordinal_cat_cols:
    print(f'col name {col} : {temp_df[col].unique()}')

col name traffic : ['high' 'jam' 'low' 'medium']
col name distance_type : ['short' 'very_long' 'medium' 'long']


In [16]:
# generate order for ordinal encoding

traffic_order = ["low","medium","high","jam"]

distance_type_order = ["short","medium","long","very_long"]

In [17]:
preprocessor = ColumnTransformer(
    [
        ('scaling',MinMaxScaler(),num_cols),
        ('OneHotEncoder',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'),nominal_cat_cols),
        ('OrdinalEncoding',OrdinalEncoder(categories=[traffic_order,distance_type_order],handle_unknown='use_encoded_value',unknown_value=-1),ordinal_cat_cols)
    ],
    remainder='passthrough',verbose_feature_names_out=False,n_jobs=-1
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scaling', ...), ('OneHotEncoder', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and 

In [18]:
# build model Pipeline

pipeline = Pipeline(
    [
        ('preprocessor',preprocessor)
    ]
)

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scaling', ...), ('OneHotEncoder', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [19]:
X_train_trans = pipeline.fit_transform(X_train)
X_test_trans = pipeline.transform(X_test)

X_train_trans.sample(2)

,age,ratings,pickup_time_minutes,distance,weather_fog,weather_sandstorms,weather_stormy,weather_sunny,weather_windy,type_of_order_drinks,...,city_type_semi-urban,city_type_urban,is_weekend_1,order_time_of_day_evening,order_time_of_day_morning,order_time_of_day_night,traffic,distance_type,vehicle_condition,multiple_deliveries
24593,0.526316,0.96,0.5,0.561656,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,2.0,2,1.0
44792,0.894737,0.80,0.0,0.243669,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2,0.0


## Optuna HyperParameter tuning

In [20]:
dagshub.init(repo_owner='jay-kanakia', repo_name='swiggy_delivery_time_prediction', mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow')
mlflow.set_experiment(experiment_name='Exp 2 : Model Selection with Optuna HP')

Accessing as jay-kanakia

Initialized MLflow to track repo "jay-kanakia/swiggy_delivery_time_prediction"

Repository jay-kanakia/swiggy_delivery_time_prediction initialized!

<Experiment: artifact_location='mlflow-artifacts:/ba2bb6f5d73b429eb2b858c065e71ecc', creation_time=1775374351700, experiment_id='3', last_update_time=1775374351700, lifecycle_stage='active', name='Exp 2 : Model Selection with Optuna HP', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [21]:
def objective(trial):

    with mlflow.start_run(nested=True):

        model_name = trial.suggest_categorical('model',['RF','GB','XGB','LGBM'])

        if model_name == 'SVM':
            kernel_svm = trial.suggest_categorical('kernel_svm',['linear','poly','rbf'])
            
            if kernel_svm == 'linear':
                c_linear = trial.suggest_float('c_linear',0.001,100,log=True)
                model = SVR(kernel = 'linear',C = c_linear)

            elif kernel_svm == 'poly':
                c_poly = trial.suggest_float('c_poly',0,10)
                degree_poly = trial.suggest_int('degree_poly',1,5)
                model = SVR(kernel='poly',degree=degree_poly,C = c_poly)

            else:
                c_rbf = trial.suggest_float('c_rbf',0,10)
                gamma_rbf = trial.suggest_float('gamma_rbf',0.0001,10,log=True)
                model = SVR(kernel = 'rbf',C = c_rbf,gamma = gamma_rbf)

        elif model_name == 'RF':
            n_estimator_rf = trial.suggest_int('n_estimator_rf',10,200)
            max_depth_rf = trial.suggest_int('max_depth_rf',2,20)
            model = RandomForestRegressor(n_estimators=n_estimator_rf,max_depth=max_depth_rf,random_state=42,n_jobs=-1)

        elif model_name == 'GB':
            n_estimator_gb = trial.suggest_int('n_estimator_gb',10,200)
            learning_rate_gb  = trial.suggest_float('learning_rate_gb',0.01,0.3)
            max_depth_gb = trial.suggest_int('max_depth_gb',2,20)
            model = GradientBoostingRegressor(n_estimators=n_estimator_gb,learning_rate=learning_rate_gb,max_depth=max_depth_gb,random_state=42)

        elif model_name == 'XGB':
            n_estimator_xgb = trial.suggest_int('n_estimator_xgb',10,200)
            learning_rate_xgb  = trial.suggest_float('learning_rate_xgb',0.01,0.3)
            max_depth_xgb = trial.suggest_int('max_depth_xgb',2,20)
            model = XGBRegressor(n_estimators=n_estimator_xgb,learning_rate=learning_rate_xgb,max_depth=max_depth_xgb,random_state=42)

        elif model_name == 'LGBM':
            n_estimator_lgbm = trial.suggest_int('n_estimator_lgbm',10,200)
            learning_rate_lgbm = trial.suggest_float('learning_rate_lgbm', 0.01, 0.1)
            max_depth_lgbm = trial.suggest_int('max_depth_lgbm',2,20)
            num_leaves = trial.suggest_int('num_leaves', 20, 150)
            min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 10, 50)
            model = LGBMRegressor(n_estimators=n_estimator_lgbm,learning_rate=learning_rate_lgbm,max_depth=max_depth_lgbm,random_state=42,n_jobs=-1)

        elif model_name == 'KNN':
            n_neighbors_knn = trial.suggest_int('n_neighbors_knn',1,25)
            weight_knn = trial.suggest_categorical('weight_knn',['uniform','distance'])
            model = KNeighborsRegressor(n_neighbors=n_neighbors_knn,weights=weight_knn)


        # train model
        model.fit(X_train_trans,y_train_pt)

        # log model params
        mlflow.log_params(model.get_params())

        # get prediction
        y_pred_train = model.predict(X_train_trans)
        y_pred_test  = model.predict(X_test_trans)

        # get the actual prediction values
        y_pred_train_org = pt.inverse_transform(y_pred_train.reshape(-1,1))
        y_pred_test_org = pt.inverse_transform(y_pred_test.reshape(-1,1))

        # Calculate error
        test_error = mean_absolute_error(y_test,y_pred_test_org)
        train_error = mean_absolute_error(y_train,y_pred_train_org)

        mlflow.log_metric('train_mae',train_error)
        mlflow.log_metric('test_mae',test_error)

    return test_error

In [22]:
study = optuna.create_study(direction='minimize',study_name = 'model_selection')

with mlflow.start_run(run_name='Best Model'):
    
    study.optimize(objective,n_trials=30)

    mlflow.log_params(study.best_params)

    mlflow.log_metric('best_score',study.best_value)

[I 2026-04-05 14:27:15,411] A new study created in memory with name: model_selection


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001949 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 352
[LightGBM] [Info] Number of data points in the train set: 30156, number of used features: 25
[LightGBM] [Info] Start training from score -0.000000
🏃 View run amusing-skunk-439 at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/3/runs/ef8561dafb8843ceb0de44f38baf3759
🧪 View experiment at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/3


[I 2026-04-05 14:27:20,420] Trial 0 finished with value: 3.072000508580551 and parameters: {'model': 'LGBM', 'n_estimator_lgbm': 160, 'learning_rate_lgbm': 0.03563467926965561, 'max_depth_lgbm': 18, 'num_leaves': 105, 'min_data_in_leaf': 15}. Best is trial 0 with value: 3.072000508580551.


🏃 View run unequaled-worm-719 at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/3/runs/87fd414f34ef43f296a92023b0f65cd7
🧪 View experiment at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/3


[I 2026-04-05 14:27:24,812] Trial 1 finished with value: 3.1151108741760254 and parameters: {'model': 'XGB', 'n_estimator_xgb': 131, 'learning_rate_xgb': 0.2658735378063197, 'max_depth_xgb': 7}. Best is trial 0 with value: 3.072000508580551.


...

🏃 View run abrasive-dove-903 at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/3/runs/1307580b7b954b21891e6d317c307b1f
🧪 View experiment at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/3


[I 2026-04-05 14:31:23,302] Trial 28 finished with value: 3.2138681411743164 and parameters: {'model': 'XGB', 'n_estimator_xgb': 55, 'learning_rate_xgb': 0.1346035983479103, 'max_depth_xgb': 15}. Best is trial 22 with value: 3.0175349712371826.


🏃 View run merciful-stag-41 at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/3/runs/b4d115929c9d442db6f14bc11522a3c6
🧪 View experiment at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/3


[I 2026-04-05 14:31:28,926] Trial 29 finished with value: 3.0389769077301025 and parameters: {'model': 'XGB', 'n_estimator_xgb': 91, 'learning_rate_xgb': 0.050408525596693995, 'max_depth_xgb': 11}. Best is trial 22 with value: 3.0175349712371826.


🏃 View run Best Model at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/3/runs/cc954d6638a14d0192944513bff3f993
🧪 View experiment at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/3


In [23]:
study.best_params

{'model': 'XGB',
 'n_estimator_xgb': 68,
 'learning_rate_xgb': 0.0724477304551672,
 'max_depth_xgb': 10}

In [25]:
study.best_value

3.0175349712371826

In [26]:
xgb_params = {'n_estimator_xgb': 68,
 'learning_rate_xgb': 0.0724477304551672,
 'max_depth_xgb': 10}

In [27]:
xgb = XGBRegressor(**xgb_params)
xgb.fit(X_train_trans,y_train_pt)

c:\Users\Jay Kanakia\Desktop\CampusX\Projects\swiggy_delivery_time_prediction\myenv\Lib\site-packages\xgboost\training.py:200: UserWarning: [14:38:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "learning_rate_xgb", "max_depth_xgb", "n_estimator_xgb" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [29]:
# get the predictions
y_pred_train = xgb.predict(X_train_trans)
y_pred_test = xgb.predict(X_test_trans)

In [30]:
# get the actual predictions values

y_pred_train_org = pt.inverse_transform(y_pred_train.reshape(-1,1))
y_pred_test_org = pt.inverse_transform(y_pred_test.reshape(-1,1))

In [31]:
from sklearn.metrics import mean_absolute_error, r2_score

print(f"The train error is {mean_absolute_error(y_train,y_pred_train_org):.2f} minutes")
print(f"The test error is {mean_absolute_error(y_test,y_pred_test_org):.2f} minutes")

The train error is 2.63 minutes
The test error is 3.08 minutes


In [32]:
print(f"The train r2 score is {r2_score(y_train,y_pred_train_org):.2f}")
print(f"The test r2 score is {r2_score(y_test,y_pred_test_org):.2f}")

The train r2 score is 0.88
The test r2 score is 0.83


In [34]:
# dataframe of results

study.trials_dataframe().sample(2)

,number,value,datetime_start,datetime_complete,duration,params_learning_rate_gb,params_learning_rate_lgbm,params_learning_rate_xgb,params_max_depth_gb,params_max_depth_lgbm,params_max_depth_rf,params_max_depth_xgb,params_min_data_in_leaf,params_model,params_n_estimator_gb,params_n_estimator_lgbm,params_n_estimator_rf,params_n_estimator_xgb,params_num_leaves,state
14,14,3.488349,2026-04-05 14:30:00.902823,2026-04-05 14:30:04.245724,0 days 00:00:03.342901,NaN,0.032309,NaN,NaN,4.0,NaN,NaN,15.0,LGBM,NaN,165.0,NaN,NaN,120.0,COMPLETE
28,28,3.213868,2026-04-05 14:31:16.949354,2026-04-05 14:31:23.302102,0 days 00:00:06.352748,NaN,NaN,0.134604,NaN,NaN,NaN,15.0,NaN,XGB,NaN,NaN,NaN,55.0,NaN,COMPLETE


In [36]:
study.trials_dataframe()['params_model'].value_counts()

params_model
XGB     17
LGBM     7
GB       3
RF       3
Name: count, dtype: int64

In [40]:
study.trials_dataframe().groupby(['params_model'])['value'].mean().sort_values()

params_model
LGBM    3.188129
GB      3.213849
XGB     3.314212
RF      4.157642
Name: value, dtype: float64

In [42]:
score  = cross_val_score(xgb,X_train_trans,y_train_pt,scoring='neg_mean_absolute_error',cv=5,n_jobs=-1)
score

array([-0.35230818, -0.34758249, -0.34491307, -0.34600523, -0.34557638])

In [44]:
model = TransformedTargetRegressor(regressor=xgb,transformer=pt)

In [47]:
score  = cross_val_score(model,X_train_trans,y_train,scoring='neg_mean_absolute_error',cv=5,n_jobs=-1)
score

array([-3.20069432, -3.13977528, -3.13746285, -3.12068963, -3.12642002])

### Visualization

In [48]:
optuna.visualization.plot_optimization_history(study)

In [49]:
# parallel coordinate plot

optuna.visualization.plot_parallel_coordinate(study,params=["model"])